In [2]:
import pandas as pd
from scipy import stats

# =====================================================================
# 1. CARGA DE DATOS (Reemplaza con tus archivos reales)
# =====================================================================
df_n8n = pd.read_csv("1.csv")
df_lc = pd.read_csv("2.csv")

# Unimos por pregunta para garantizar el emparejamiento exacto
df_merged = pd.merge(df_n8n, df_lc, on="Question", suffixes=("_n8n", "_lc"))

# =====================================================================
# 2. CONFIGURACIÓN DE LAS MÉTRICAS DE LA TABLA LATEX
# =====================================================================
# Diccionario con las métricas a evaluar: 
# (Nombre en consola, Columna en CSV, ¿Menor es mejor?)
metricas_tabla = [
    # Token consumption
    ("Prompt tokens", "Prompt Tokens", True),
    ("Completion tokens", "Completion Tokens", True),
    ("Total tokens", "Total Tokens", True),
    ("Economic Efficiency", "economic_eff_raw", False),  # Mayor puntaje/$ es mejor
    
    # Latency
    ("Execution time (ms)", "Execution Time (ms)", True),
    
    # Response quality
    ("RAGking Score", "RAGking_score", False),
    ("Avg. correctness", "Correctness", False),
    ("Cosine similarity", "String Similarity", False),
    
    # Operational Cost
    ("Cost per query", "cost_per_query", True)
]

# =====================================================================
# 3. MOTOR DE ANÁLISIS WILCOXON
# =====================================================================
resultados = []

print(f"{'Métrica':<25} | {'p-value':<10} | {'Significativo?':<15} | {'Ganador Estadístico'}")
print("-" * 75)

for nombre_mostrar, columna_csv, menor_es_mejor in metricas_tabla:
    col_n8n = f"{columna_csv}_n8n"
    col_lc = f"{columna_csv}_lc"
    
    # Limpiamos NaNs para esta métrica específica
    datos = df_merged[[col_n8n, col_lc]].dropna()
    a_n8n = datos[col_n8n]
    a_lc = datos[col_lc]
    
    # Ejecutamos Wilcoxon
    stat, p_value = stats.wilcoxon(a_n8n, a_lc, zero_method='wilcox', alternative='two-sided')
    
    # Determinamos significancia y ganador
    es_significativo = p_value < 0.05
    ganador = "Empate / Ninguno"
    
    if es_significativo:
        if menor_es_mejor:
            ganador = "n8n" if a_n8n.mean() < a_lc.mean() else "LangChain"
        else:
            ganador = "n8n" if a_n8n.mean() > a_lc.mean() else "LangChain"
            
    # Formateo para la consola
    sig_texto = "✅ SÍ" if es_significativo else "❌ NO"
    print(f"{nombre_mostrar:<25} | {p_value:<10.6f} | {sig_texto:<15} | {ganador}")
    
    # Guardamos para posible exportación
    resultados.append({
        "Metric": nombre_mostrar,
        "p-value": p_value,
        "Significant": es_significativo,
        "Winner": ganador
    })

print("-" * 75)
print("Nota: Las métricas de 'Deployment' (Deploy Time, Difficulty, Maintenance)")
print("fueron excluidas porque son valores estáticos globales (N=1), no iterativos.")

Métrica                   | p-value    | Significativo?  | Ganador Estadístico
---------------------------------------------------------------------------
Prompt tokens             | 0.000000   | ✅ SÍ            | LangChain
Completion tokens         | 0.000000   | ✅ SÍ            | LangChain
Total tokens              | 0.000000   | ✅ SÍ            | LangChain
Economic Efficiency       | 0.000000   | ✅ SÍ            | LangChain
Execution time (ms)       | 0.000000   | ✅ SÍ            | LangChain
RAGking Score             | 0.000000   | ✅ SÍ            | LangChain
Avg. correctness          | 0.613489   | ❌ NO            | Empate / Ninguno
Cosine similarity         | 0.080548   | ❌ NO            | Empate / Ninguno
Cost per query            | 0.000000   | ✅ SÍ            | LangChain
---------------------------------------------------------------------------
Nota: Las métricas de 'Deployment' (Deploy Time, Difficulty, Maintenance)
fueron excluidas porque son valores estáticos globales (N=1)